# EY Open Science AI & Data Challenge 2026
## Modeling Notebook — Training, Evaluation & Submission

| | |
|---|---|
| **Notebook purpose** | Per-target ensemble training, features importances analysis, and final submission |
| **Input** | `train_preprocessed.csv`, `submission_preprocessed.csv` |
| **Output** | `final_submission.csv` |

---

### Ensemble architecture

| Target | Strategy | Blend weights |
|---|---|---|
| Total Alkalinity | ExtraTrees + RandomForest + HGBM | 45 / 45 / 10 |
| Electrical Conductance | RandomForest + XGBoost + HGBM | 40 / 30 / 30 |
| Dissolved Reactive Phosphorus | RandomForest only | — |

### Best leaderboard R² = 0.4829


---
## 1. Imports

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
import shap

RANDOM_STATE = 42
print('All imports successful.')


---
## 2. Load Preprocessed Data

Load the outputs of `EY_Preprocessing_Notebook.ipynb`.
All imputation, distance-to-sea computation, and feature engineering
have already been applied — no further transformation is needed here.


In [ ]:
DATA_DIR = '.'  # update if files are in a different directory

target_columns = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']

train_feat      = pd.read_csv(os.path.join(DATA_DIR, 'train_preprocessed.csv'))
submission_feat = pd.read_csv(os.path.join(DATA_DIR, 'submission_preprocessed.csv'))

feature_columns = [
    c for c in train_feat.columns
    if c not in target_columns + ['Sample Date']
]

print(f'Training set:    {train_feat.shape[0]:,} rows x {train_feat.shape[1]} columns')
print(f'Submission set:  {submission_feat.shape[0]:,} rows x {submission_feat.shape[1]} columns')
print(f'Feature columns: {len(feature_columns)}')
print(f'Target columns:  {target_columns}')


---
## 3. Per-Target Feature Sets

Each target uses a curated feature subset selected through recursive feature
elimination and domain reasoning. Separate feature sets prevent irrelevant features
from introducing noise for a given indicator.


In [ ]:
# NOTE — REDACTED FOR THIS PUBLIC REPOSITORY.
# The exact per-target feature sets used in our competition entry (35-42
# features per target, selected via recursive feature addition validated
# on the leaderboard) have been withheld, as this is part of ongoing
# research we intend to continue. See docs/Model_Description.md for the
# general feature engineering categories used.
#
# To reproduce structurally, feature_sets would map each target column to
# a curated list of column names from `feature_columns` (defined above).

feature_sets = {
    target: feature_columns for target in target_columns
}

print('NOTE: using the full feature set as a placeholder for the redacted, '
      'competition-specific per-target feature selections.')
for target, feats in feature_sets.items():
    print(f'  {target}: {len(feats)} features (placeholder — see note above)')


---
## 4. Model Training

Each target is trained with an independent ensemble strategy.
Predictions are clipped to 0 (water quality concentrations cannot be negative).


In [ ]:
# NOTE — REDACTED FOR THIS PUBLIC REPOSITORY.
# The exact hyperparameters (n_estimators, max_depth, learning_rate, etc.)
# and ensemble blend weights used in our competition entry have been
# withheld, as this is part of ongoing research we intend to continue.
#
# Structurally, the approach trains one independent ensemble per target:
#   - Total Alkalinity:            ExtraTrees + RandomForest + HistGradientBoosting
#   - Electrical Conductance:      RandomForest + XGBoost + HistGradientBoosting
#   - Dissolved Reactive Phosphorus: RandomForest (standalone)
# with predictions clipped to zero and blend weights tuned on the leaderboard.
# See docs/Model_Description.md for the full methodology description.

models = {}
predictions = {}

for target in target_columns:
    print(f"\n🚀 Training model for {target}")

    cols = feature_sets.get(target, feature_columns)
    X_tr = train_feat[cols]
    y_tr = train_feat[target]
    X_val = submission_feat[cols]

    # Placeholder: hyperparameters and ensemble composition redacted.
    # Substitute your own tuned models/weights here to reproduce structurally.
    model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_val)
    preds = np.clip(preds, a_min=0, a_max=None)

    models[target] = model
    predictions[target] = preds

print('\nAll models trained (placeholder configuration — see note above).')


---
## 5. Results & Evaluation

### 5.1 Feature Importance


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for ax, target in zip(axes, target_columns):
    cols = feature_sets[target]
    imp  = pd.Series(models[target].feature_importances_, index=cols).sort_values(ascending=True)
    imp.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(target, fontsize=10, fontweight='bold')
    ax.set_xlabel('Feature Importance')
    ax.tick_params(axis='y', labelsize=8)
plt.suptitle('Feature Importances by Target', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


### 5.2 SHAP Analysis

SHAP (SHapley Additive exPlanations) decomposes each prediction into per-feature
additive contributions. The bar chart ranks features by mean absolute SHAP value;
the beeswarm plot additionally shows the direction and magnitude of each feature's
effect across the sample.

NB:that take long time to execute


In [ ]:
for target in target_columns:
    print(f'\n{"─"*55}')
    print(f'  SHAP Analysis: {target}')
    print(f'{"─"*55}')
    cols    = feature_sets[target]
    X_shap  = train_feat[cols].sample(min(500, len(train_feat)), random_state=RANDOM_STATE)
    explainer   = shap.TreeExplainer(models[target])
    shap_values = explainer.shap_values(X_shap)

    shap.summary_plot(shap_values, X_shap, plot_type='bar', show=False, max_display=15)
    plt.title(f'SHAP Feature Importance — {target}')
    plt.tight_layout(); plt.show()

    shap.summary_plot(shap_values, X_shap, show=False, max_display=15)
    plt.title(f'SHAP Beeswarm — {target}')
    plt.tight_layout(); plt.show()


---
## 6. Final Submission


In [ ]:
submission_template = pd.read_csv('submission_template.csv')
submission_template.head()


In [ ]:
submission_out = submission_template[['Latitude', 'Longitude', 'Sample Date']].copy()
for target in target_columns:
    submission_out[target] = predictions[target]

submission_out.to_csv('final_submission.csv', index=False)
print('Saved: final_submission.csv')
print()
submission_out.head()


---
## 7. Conclusion & Lessons Learned

### What Worked Best

1. **Hierarchical spatio-temporal imputation** — preserving spatial and temporal structure
   during imputation was crucial, especially for remote sensing features with patchy cloud coverage.

2. **Per-target models** — the three water quality indicators respond to entirely different
   environmental drivers. A single multi-output model was consistently outperformed by per-target models.

3. **Rich feature engineering** — particularly precipitation rolling windows (multiple time scales),
   PET×coordinate interactions, and terrain-based regime features (TWI groups, rain regime).
   These physically-motivated features replaced raw coordinates and reduced spatial overfitting.

4. **Weighted ensembles** — combining ExtraTrees, RandomForest, XGBoost and HistGradientBoosting
   with target-specific weights gave consistent improvements over any single model.

### The Core Challenge: Spatial Generalization

The most important lesson is that the **geographic gap between training and submission stations**
is the dominant difficulty. Submission stations are clustered along South Africa's southern coast
while training stations span the entire country. This mismatch makes standard cross-validation
misleadingly optimistic and rewards features that encode physical processes rather than location.

### What We Would Explore With More Time

- **Stacking with a spatial meta-learner** trained on out-of-fold predictions
- **Richer satellite composites** — cloud-free composites over longer windows for the eastern Cape
- **Bayesian hyperparameter optimisation** (Optuna) to replace manual tuning
- **Semi-supervised learning** — using submission station features (without labels) to
  align training and submission distributions

### Final Score

> **Best Leaderboard R² = 0.4829**
